In [ ]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
import os
import torch
from torch.utils.data import Dataset
import torchvision
import torchvision.transforms as transforms
import pandas as pd
from vit import ViTDecoder as vit
from PIL import Image
import math as math
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models
import numpy as np

In [ ]:
# Custom Padding Transformer (FIX THIS)
class PadToSize:
    def __init__(self, target):
        self.target = target

    def __call__(self, img):
        # Calculate padding sizes

        colors, width, height = img.size()
        pad_left = math.floor((max(0, self.target[0] - width)/2))
        pad_right = math.ceil(((max(0, self.target[0]-width))/2))-1
        pad_top = math.floor((max(0, self.target[1] - height)/2))+1
        pad_bot = math.ceil((max(0,self.target[1] - height)/2))

        padding = (pad_left, pad_top, pad_right, pad_bot)
        if self.target[0]-width > width or self.target[1]-height > height:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='constant')
        else:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='reflect')
        
class ConvertToFloat32(object):
    def __call__(self, tensor):
        return tensor.to(torch.float32)

In [ ]:
class StackedImagesDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = sorted(os.listdir(image_dir))  # Sorted to align image order
        self.label_files = sorted(os.listdir(label_dir))  # Assuming labels have the same order
        self.transform = transform
        self.current_transform = transforms.Compose([
            transforms.Normalize(mean=[0.2693, 0.1058, 0.3865], std=[0.0505, 0.1627, 0.0643])  # Normalize
            ])
        self.eff_dist_transform = transforms.Compose([
            transforms.Normalize(mean=[0.1720, 0.6108, 0.5125], std=[0.0962, 0.1022, 0.0644])  # Normalize
            ])
        self.pdn_density_transform = transforms.Compose([
            transforms.Normalize(mean=[0.3741, 0.4506, 0.3820], std=[0.2636, 0.2681, 0.1267])  # Normalize
            ])
        self.ir_drop_transform = transforms.Compose([
            transforms.Normalize(mean=[0.2233, 0.3428, 0.5156], std=[0.0592, 0.1559, 0.0490])  # Normalize
            ])

    
    def __len__(self):
        return len(self.label_files) # based off of the label amount
    
    def __getitem__(self, idx):
        # Get 3 consecutive images for stacking (you could choose any other strategy here)
        names = []

        img1_path = os.path.join(self.image_dir, self.image_files[3*idx])
        img2_path = os.path.join(self.image_dir, self.image_files[3*idx+1])
        img3_path = os.path.join(self.image_dir, self.image_files[3*idx+2])
        
        # Load images
        img1 = read_image(img1_path)
        img2 = read_image(img2_path)
        img3 = read_image(img3_path)
        if self.transform:
            img1 = self.transform(img1)
            #img1 = self.current_transform(img1)
            img2 = self.transform(img2)
            #img2 = self.eff_dist_transform(img2)
            img3 = self.transform(img3)
            #img3 = self.pdn_density_transform(img3)


        # Stack the 3 images along the channel dimension (depth-wise)
        stacked_images = torch.cat([img1,img2,img3])
        
        # Load corresponding label image
        label_path = os.path.join(self.label_dir, self.label_files[idx])  # Label corresponding to last image in stack
        label = read_image(label_path)
        if self.transform:
            label = self.transform(label)  # Apply transformation if needed
            #label = self.ir_drop_transform(label)
        
        names.append(self.image_files[3*idx])
        names.append(self.image_files[3*idx+1])
        names.append(self.image_files[3*idx+2])
        names.append(self.label_files[idx])

        return stacked_images, label #names #, idx, names

In [ ]:
pipeline= transforms.Compose([
    # transforms.Resize((224, 224)),
    transforms.Lambda(lambda x: x[:3]),
    PadToSize((930,930)),  # Pad the image to the target size,
    ConvertToFloat32()
    #transforms.ToTensor()
])

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
import torch
from PIL import Image

pipe = StableDiffusionImg2ImgPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")

init_image = Image.open("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input/00_current.png").convert("RGB").resize((512, 512))

# Optionally set prompt for conditioning (can be blank or "IR drop heatmap")
prompt = "IR drop heatmap"

output = pipe(prompt=prompt, image=init_image, strength=0.8, guidance_scale=7.5).images[0]
output.save("ir_drop_output.png")

In [ ]:
from transformers import ViTModel, ViTConfig

config = ViTConfig.from_pretrained("google/vit-base-patch16-224-in21k")
config.num_channels = 9  # your 3 input maps
config.image_size = 930
config.patch_size = 15

vit = ViTModel(config)
import torch.nn as nn

class IRDropModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ViTModel(config)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(768, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(pixel_values=x).last_hidden_state
        patch_tokens = x[:, 1:, :]  # Remove CLS token → [B, 196, 768]
    
        # B, N, C = patch_tokens.shape  # B=batch, N=196, C=768
        # H = W = int(N ** 0.5)         # Assumes square patches, H=W=14 here

        x = patch_tokens.permute(0, 2, 1) #.reshape(-1, 768, 58, 58)  # reshape to image-like
        #return self.decoder(x)
        return x

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

dataset = StackedImagesDataset(image_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input", label_dir= "/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/ir_drop", transform=pipeline)
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])
train_dataloader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model = IRDropModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(1):
    for batch_idx, (inp, out) in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch + 1}", ncols=100)):
        # print(inp.size())
        pred = model(inp)
        print(pred.size())
        loss = criterion(pred, out)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch}, Batch {i}, Loss {loss.item()}")
